## Iowa Optimization — Hierarchical M Experiments COUNTY

This notebook runs multi-district MIP optimization experiments for county level

| Metric | Edge weight | M |
|---|---|---|
| `hop_M` | 1 + {0, M, M²} | number of nodes |
| `euclidean_M` | eucl(i,j) + {0, M, M²} | two-sweep diameter |

The penalty tier is determined by GEOID20 prefix matching:
- `GEOID20[:11]` match → same tract (no penalty)
- `GEOID20[:5]` match → same county, different tract (+M)
- Otherwise → different county (+M²)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT_DIR))

import os

from src.read import read_graph_from_json
from src.experiment import run_optimization_experiment

os.makedirs("../../results", exist_ok=True)

### Experiment parameters


In [3]:
GRAPH_LEVELS = ["county"]
DISTANCE_METRICS = ["euclidean", "hop"]

K = 4
DEVIATIONS = [50]
CONTIGUITY = [None, "tree", "dist", "dag", "cut"]
TIME_LIMIT = 3600

OBJECTIVES = ["euclidean_moi", "cut_edges"]

total_combinations = len(GRAPH_LEVELS) * len(DISTANCE_METRICS)
print(f"Graph levels     : {GRAPH_LEVELS}")
print(f"Distance metrics : {DISTANCE_METRICS}")
print(f"Combinations     : {total_combinations}")
print(f"Experiments each : {len(DEVIATIONS) * len(CONTIGUITY)} × objectives")

Graph levels     : ['county']
Distance metrics : ['euclidean', 'hop']
Combinations     : 2
Experiments each : 5 × objectives


### Run all combinations

In [4]:
for graph_level in GRAPH_LEVELS:
    print(f"Loading graph: IA_{graph_level}.json")
    G = read_graph_from_json(f"../../data/IA_{graph_level}.json", state="IA")
    has_coords = "X" in G.nodes[next(iter(G.nodes))]
    coord_key = "coords" if has_coords else "no_coords"
    print(
        f"  {G.number_of_nodes()} nodes, {G.number_of_edges()} edges | coords: {has_coords}"
    )

    for dist_metric in DISTANCE_METRICS:
        label = f"IA_{graph_level}_{dist_metric}"
        results_file = f"../../results/{label}_optimization.csv"

        df = run_optimization_experiment(
            G_base=G,
            deviations=DEVIATIONS,
            contiguity_models=CONTIGUITY,
            objectives=OBJECTIVES,
            k=K,
            distance_metric=dist_metric,
            time_limit=TIME_LIMIT,
            results_file=results_file,
        )

        df["graph_level"] = graph_level
        df["distance_metric"] = dist_metric

print(" All combinations complete.")

Loading graph: IA_county.json
  99 nodes, 222 edges | coords: True
Selected roots: [56, 26, 40, 27]
Distance metric: euclidean
Setting Euclidean edge weights...

Total experiments: 10
Results will be saved to: ../../results/IA_county_euclidean_optimization.csv

[1/10] deviation=50 | contiguity=None (dist: N/A) | objective_type=euclidean_moi
Using L, U, k = 797543 797642 4
Set parameter Username
Academic license - for non-commercial use only - expires 2026-11-24
Set parameter OutputFlag to value 1
Set parameter LogToConsole to value 1
Set parameter MIPGap to value 0
Set parameter PoolSearchMode to value 0
Set parameter PoolSolutions to value 1
Set parameter TimeLimit to value 3600
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 22.6.0 22H730)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  3600
MIPGap  0
PoolSolutions  1

Optimize a model with 107 rows, 396 columns and 1188 nonzero

In [5]:
df

,time_best,objective_type,objective,obj_bound,obj_gap,nonzeros,num_solutions,status,contiguity,sparsity,num_callbacks,num_lazy_cuts,bnb_nodes,roots,districts,deviation,contiguity_distance,k,distance_metric,graph_level
0,1.132701,euclidean_moi,7.800526e+06,7.800526e+06,0.000000e+00,1188,1,Optimal,NaN,97.196262,0,0,8897.0,"[56, 26, 40, 27]","[[5, 7, 10, 11, 13, 15, 16, 29, 34, 51, 56, 61...",50,N/A,4,hop,county
1,3768.037519,cut_edges,4.800000e+01,3.300000e+01,3.125000e-01,8736,1,Time Limit,NaN,99.843183,0,0,544044.0,"[56, 26, 40, 27]","[[4, 8, 10, 18, 31, 42, 56], [3, 6, 12, 24, 26...",50,N/A,4,hop,county
2,1.092257,euclidean_moi,NaN,inf,NaN,1972,0,Infeasible,tree,99.002044,0,0,19440.0,"[56, 26, 40, 27]",None,50,hop,4,hop,county
3,2.700102,cut_edges,NaN,inf,NaN,9520,0,Infeasible,tree,99.853747,0,0,7434.0,"[56, 26, 40, 27]",None,50,hop,4,hop,county
4,377.319828,euclidean_moi,8.261284e+06,8.261284e+06,6.764004e-16,2228,1,Optimal,dist,98.872492,0,0,982525.0,"[56, 26, 40, 27]","[[5, 6, 7, 10, 11, 13, 15, 16, 29, 45, 50, 51,...",50,hop,4,hop,county
5,2376.063518,cut_edges,5.100000e+01,5.100000e+01,0.000000e+00,9776,1,Optimal,dist,99.849815,0,0,1154413.0,"[56, 26, 40, 27]","[[5, 7, 11, 13, 15, 25, 29, 36, 45, 47, 50, 51...",50,hop,4,hop,county
6,13.701187,euclidean_moi,7.972387e+06,7.972387e+06,0.000000e+00,2468,1,Optimal,dag,98.751037,0,0,122937.0,"[56, 26, 40, 27]","[[5, 6, 7, 10, 11, 15, 16, 29, 34, 45, 50, 51,...",50,hop,4,hop,county
7,4235.785798,cut_edges,5.200000e+01,3.700000e+01,2.884615e-01,10016,1,Time Limit,dag,99.846128,0,0,244846.0,"[56, 26, 40, 27]","[[3, 6, 11, 15, 28, 29, 34, 45, 50, 51, 56, 61...",50,hop,4,hop,county
8,1387.898722,euclidean_moi,7.963866e+06,7.963866e+06,0.000000e+00,1188,1,Optimal,cut,97.196262,82,675,6214963.0,"[56, 26, 40, 27]","[[5, 6, 7, 10, 11, 15, 16, 29, 34, 45, 50, 51,...",50,N/A,4,hop,county
9,3635.733201,cut_edges,NaN,3.400000e+01,NaN,8736,0,Time Limit,cut,99.843183,43,645,1031812.0,"[56, 26, 40, 27]",None,50,N/A,4,hop,county
